# PRED horizon/task summary

Compact notebook for the manuscript table:

`Horizon x Supervised ML Task x Target Variable x paper counts x early/actionable prediction x implemented intervention/deployment`

The reusable classification and aggregation logic lives in `notion_zotero.analysis.pred_horizon_summary`.


## Working definitions

- **Short-term**: course-level prediction, including course grades, scores, exams, pass/fail, assignments, quizzes, modules, activities, and next-question outcomes.
- **Long-term**: program-level prediction, including program retention, persistence, graduation, degree completion, transfer, enrollment, and program/institution-level dropout.
- **Reviewed overrides**: specific papers with generic target wording are assigned by paper-level review rather than by broad aliases. Generic `dropout` alone is intentionally not a long-term alias.
- **Target aliases**: obvious targets are split into literal categories such as `Course grade`, `Exam grade / score`, `GPA`, `Assessment / assignment score`, `Pass/fail`, and `Dropout / retention`; ambiguous targets remain `Other / ambiguous` for manual review.
- **Early/actionable prediction**: prediction moment occurs before, at the start of, or during the course/program rather than only after completion.
- **Implemented intervention or deployment**: explicit `Deployed by Instructor`, `Integrated in LMS`, classroom/practice deployment, or production/practice usage. `Tested on New Students`, `Deployable`, and `Prototype` alone are not counted as implemented/deployed.


In [5]:
from pathlib import Path

from notion_zotero.analysis.pred_horizon_summary import (
    build_pred_horizon_task_summary_from_canonical,
    write_pred_horizon_task_summary_workbook,
)

PULLED_DIR = Path("data/pulled/notion/learning_analytics_review")
OUT_PATH = Path("data/analysis_outputs/pred_horizon_task_target_table.xlsx")
ACCEPTED_ONLY = True


In [6]:
if not PULLED_DIR.exists() or not any(PULLED_DIR.glob("*.canonical.json")):
    raise FileNotFoundError(f"No canonical bundles found in {PULLED_DIR}. Run `notion-zotero pull-notion` first.")

target_table, paper_level_detail, _clean_logs = build_pred_horizon_task_summary_from_canonical(
    PULLED_DIR,
    accepted_only=ACCEPTED_ONLY,
)

print(f"Classified PRED paper-task-horizon rows: {len(paper_level_detail)}")
print(f"Unique classified PRED papers: {paper_level_detail['paper_id'].nunique() if not paper_level_detail.empty else 0}")
display(target_table)


Classified PRED paper-task-horizon rows: 210
Unique classified PRED papers: 134


,Horizon,Supervised ML Task,Target Variable,Number of Research Papers,Papers with early/actionable prediction,Papers with implemented intervention or deployment,Papers with at-risk framing,Raw target evidence
0,Short-term,Classification,Assessment / assignment pass/fail,3,3,1,0,Success in Lab Assessments | Pass (1) vs Fail ...
1,Short-term,Classification,Assessment / assignment score,3,0,0,0,Student Attrition | Submitting at Least 70% of...
2,Short-term,Classification,At-risk / performance tier,3,2,1,2,Obtaining a top 20% GPA on a MSc program | Top...
3,Short-term,Classification,Certification / completion,1,1,0,0,Course Completion | Certificate (1) vs No Cert...
4,Short-term,Classification,Course grade / score,23,13,0,11,End of Course Grade - Inferred from Clustering...
5,Short-term,Classification,Course pass/fail,39,20,1,0,End of Course Performance | Pass (>=5) vs Fail...
6,Short-term,Classification,Dropout / retention,14,11,0,1,Dropout | Dropout vs No Dropout | End of Cours...
7,Short-term,Classification,Exam grade / score,7,4,0,2,Course Performance on Final Exam | Top 60% vs ...
8,Short-term,Classification,Exam pass/fail,9,6,2,0,Final Exam Grade | -Fail (<5.5) vs q - Pass (≥...
9,Short-term,Classification,GPA,3,1,0,1,"End of Semester GPA | At-Risk (<60% GPA, i.e.,..."


In [ ]:

# ============================================================
# Audit: papers with no horizon + ambiguous target variables
# ============================================================
# Run this cell after the build cell above to surface any rows that
# still need manual review before publication.
import json
import pandas as pd

from notion_zotero.analysis.pred_horizon_summary import (
    classify_horizons_with_source,
    classify_supervised_ml_task,
)
from notion_zotero.analysis.summarizer import is_accepted
from notion_zotero.schemas.domain_packs.education_learning_analytics import task_label_fn

# ── 1. Papers with no horizon classification ─────────────────────────────────
# build_pred_horizon_task_detail silently drops rows where classify_horizons
# returns []. This loop collects them so you can review and fix the aliases.
_unclassified = []
for _path in sorted(PULLED_DIR.glob("*.canonical.json")):
    try:
        _bundle = json.loads(_path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if ACCEPTED_ONLY and not is_accepted(_bundle):
        continue

    # Build a set of reference_task_ids that belong to PRED tasks only.
    # This prevents KT / DESC / REC extractions from appearing in this audit.
    _task_name_by_id = {t["id"]: t.get("name", "") for t in _bundle.get("tasks", [])}
    _pred_rt_ids = {
        _rt["id"]
        for _rt in _bundle.get("reference_tasks", [])
        if task_label_fn(_task_name_by_id.get(_rt.get("task_id", ""), "")) == "PRED"
    }

    _ref_by_id = {r["id"]: r for r in _bundle.get("references", []) if r.get("id")}
    for _te in _bundle.get("task_extractions", []):
        if _te.get("reference_task_id", "") not in _pred_rt_ids:
            continue
        for _row in _te.get("extracted", []):
            if classify_supervised_ml_task(_row.get("Task")) is None:
                continue
            _ref = next(iter(_ref_by_id.values()), {})
            _title = _ref.get("title") or _row.get("source_title") or ""
            _paper_id = _ref.get("id") or ""
            _horizon_input = {**_ref, **_row, "Paper title": _title}
            if not classify_horizons_with_source(_horizon_input):
                _unclassified.append({
                    "paper_id": _paper_id,
                    "Paper title": _title,
                    "Student Performance Definition": _row.get("Student Performance Definition", ""),
                    "Target": _row.get("Target", ""),
                    "Context": _row.get("Context", ""),
                    "Courses": _row.get("Courses", ""),
                    "Moment of Prediction": _row.get("Moment of Prediction", ""),
                })

_unclassified_df = pd.DataFrame(_unclassified).drop_duplicates(
    subset=["paper_id", "Target"]
).reset_index(drop=True)

print(f"Paper/rows with NO horizon classification: {len(_unclassified_df)}")
print(f"Unique papers: {_unclassified_df['paper_id'].nunique() if not _unclassified_df.empty else 0}")
if not _unclassified_df.empty:
    display(
        _unclassified_df[[
            "Paper title", "Target", "Student Performance Definition",
            "Context", "Courses", "Moment of Prediction",
        ]]
    )

# ── 2. Papers with 'Other / ambiguous' target variable ───────────────────────
_ambiguous = (
    paper_level_detail[paper_level_detail["Target Variable"] == "Other / ambiguous"]
    .drop_duplicates(subset=["paper_id", "Raw target evidence"])
    .reset_index(drop=True)
)
print(f"\nPaper-horizon rows with 'Other / ambiguous' target: {len(_ambiguous)}")
print(f"Unique papers: {_ambiguous['paper_id'].nunique() if not _ambiguous.empty else 0}")
if not _ambiguous.empty:
    display(
        _ambiguous[[
            "Paper title", "Horizon", "Raw target evidence", "Horizon source",
        ]]
    )
else:
    print("No ambiguous target variables remain — table is publication-ready.")


Paper/rows with NO horizon classification: 23
Unique papers: 23


,Paper title,Target,Student Performance Definition,Context,Courses,Moment of Prediction
0,Using Clickstream Data Mining Techniques to Un...,,,Higher Education,1 Preparation for General Chemistry Course,
1,Semester-level Spacing but Not Procrastination...,,,Higher Education,4 editions of an Introductory Programming Course,
2,Do Not Trust a Model Because It is Confident: ...,,,Higher Education,3 Flipped Classroom Courses and 3,
3,Analyzing Student Procrastination in MOOCs: A ...,,,MOOC,1 Computer Science Course,
4,Variational Deep Knowledge Tracing for Languag...,Correct (1) vs Incorrect (0),Getting next word in the sequence correct,MOOC,"3 Languages on SLAM: English, Spanish and French",
5,Learning from Non-Assessed Resources: Deep Mul...,Normalized score (0-1),Score on next graded assessment on timestep t,MOOC,Each dataset often refers to a single course.,
6,Key factors predicting problem-based learning...,,,Higher Education,1 Online Course in Social Work and Problem-Sol...,
7,Modeling Knowledge Acquisition from Multiple L...,Above Median/Below Median,Score on Next Graded Learning Material\n,MOOC,Each dataset refers to one course,
8,Credit hours is not enough: Explaining undergr...,,,Higher Education,3724 UC Berkeley Courses,
9,Students matter the most in learning analytics...,,,Higher Education,Multiple courses in a Medical School (2014/201...,



Paper-horizon rows with 'Other / ambiguous' target: 21
Unique papers: 20


,Paper title,Horizon,Raw target evidence,Horizon source
0,Enhancing educational evaluation through predi...,Short-term,"Grade Range | A, B, C and F",reviewed paper-level override
1,Early segmentation of students according to th...,Long-term,AP Score | Performance Clusters: A to E,reviewed paper-level override
2,Predicting student performance using advanced ...,Long-term,Atudent will Complete Degree | True vs False,program-level alias
3,Identifying At-Risk Students for Early Interve...,Short-term,Probability of Failing an Upcoming Assessment ...,reviewed paper-level override
4,Prediction of Dilatory Behavior in eLearning: ...,Short-term,Delay in Submission | Number of Days of Delay,course-level alias
5,Deep learning approach for predicting universi...,Long-term,Student graduates or Drops out | Graduated (0)...,program-level alias
6,Evaluation of early student performance predic...,Short-term,"Performance Groups | Low, Medium, High",reviewed paper-level override
7,Predicting Dropout in Higher Education Based o...,Long-term,End of Program Graduation or Dropout | Dropped...,program-level alias
8,Analyzing undergraduate students' performance ...,Long-term,End of Program Mark | End of Program Passing M...,program-level alias
9,Student’s performance prediction based on an i...,Long-term,"Grade Point Average | Good (≥85), Average (≥75...",reviewed paper-level override


In [4]:
try:
    written = write_pred_horizon_task_summary_workbook(
        PULLED_DIR,
        OUT_PATH,
        accepted_only=ACCEPTED_ONLY,
    )
except PermissionError:
    fallback = OUT_PATH.with_name(f"{OUT_PATH.stem}_latest{OUT_PATH.suffix}")
    written = write_pred_horizon_task_summary_workbook(
        PULLED_DIR,
        fallback,
        accepted_only=ACCEPTED_ONLY,
    )
    print(f"{OUT_PATH} is locked; wrote fallback workbook instead.")
print(f"Wrote {written}")


Wrote data\analysis_outputs\pred_horizon_task_target_table.xlsx


## Audit notes

- Counts are unique papers within each Horizon x Supervised ML Task x Target Variable bucket.
- A paper can appear in more than one bucket if it reports multiple supervised tasks, multiple target variables, or both short-term and long-term prediction targets.
- `paper_level_detail` in the exported workbook keeps the paper-level rows behind the aggregate table for manual checking.
